# 고려기프트 발주서 — 파일명별 다중 상품 분석

## 구조 이해
- **메인 상품 행**: 품목코드가 `X`로 시작하지 않는 행
- **옵션 행**: 품목코드가 `X`로 시작하는 행 (`XP000`, `XC000`, `XG000`, `XD000` 등) → 직전 메인 상품의 **부대비용**으로 합산
- **목표**: 파일명 1개에 메인 상품이 2개 이상인 파일이 몇 개인지 파악

In [ ]:
import pandas as pd
from config import ORDERS_XLSX

df = pd.read_excel(ORDERS_XLSX)
print(f'전체 행 수: {len(df):,}')
print(f'컬럼 목록: {df.columns.tolist()}')
# df.head(10)

In [ ]:
# ★ 위 셀 결과를 보고 컬럼명 수정하세요
COL_FILE   = '파일명'
COL_CODE   = '품목코드'   # X로 시작하면 옵션
COL_ITEM   = '상품명'
COL_AMOUNT = '합계'

for col in [COL_FILE, COL_CODE, COL_ITEM]:
    assert col in df.columns, f"'{col}' 컬럼 없음. 전체 컬럼: {df.columns.tolist()}"
print('✅ 컬럼 확인 완료')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 품목코드가 X로 시작하면 옵션, 아니면 메인
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def is_option(code):
    if not isinstance(code, str):
        return False
    return code.strip().upper().startswith('X')

df['행_유형'] = df[COL_CODE].apply(lambda x: '옵션' if is_option(x) else '메인')

main_cnt   = (df['행_유형'] == '메인').sum()
option_cnt = (df['행_유형'] == '옵션').sum()
print(f'메인 상품 행: {main_cnt:,}개')
print(f'옵션 행    : {option_cnt:,}개')
print()
print('=== 옵션 행 품목코드 목록 ===')
print(df[df['행_유형']=='옵션'][COL_CODE].value_counts().to_string())

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 각 옵션 행을 직전 메인 상품에 연결
# (같은 파일명 내에서 위에서부터 순서대로 연결)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# 메인 상품 순번 부여 (파일명 내에서)
df = df.copy()
df['메인_idx'] = None

current_main = {}
main_counter = {}

for i, row in df.iterrows():
    fname = row[COL_FILE]
    if row['행_유형'] == '메인':
        main_counter[fname] = main_counter.get(fname, 0) + 1
        current_main[fname] = (row[COL_ITEM], main_counter[fname])
        df.at[i, '메인_idx'] = main_counter[fname]
    else:  # 옵션
        if fname in current_main:
            df.at[i, '메인_idx'] = current_main[fname][1]

print('메인 순번 부여 완료')
df[[COL_FILE, COL_ITEM, '행_유형', '메인_idx']].head(30)

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 메인 상품별 부대비용(옵션 행 금액 합산) 계산
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

if COL_AMOUNT and COL_AMOUNT in df.columns:
    df[COL_AMOUNT] = pd.to_numeric(df[COL_AMOUNT], errors='coerce').fillna(0)

    # 옵션 행의 금액을 (파일명, 메인_idx) 기준으로 합산
    option_sum = (
        df[df['행_유형'] == '옵션']
        .groupby([COL_FILE, '메인_idx'])[COL_AMOUNT]
        .sum()
        .reset_index()
        .rename(columns={COL_AMOUNT: '부대비용'})
    )

    # 메인 상품 행에 병합
    main_df = df[df['행_유형'] == '메인'].copy()
    main_df = main_df.merge(option_sum, on=[COL_FILE, '메인_idx'], how='left')
    main_df['부대비용'] = main_df['부대비용'].fillna(0)
    main_df['총비용'] = main_df[COL_AMOUNT] + main_df['부대비용']

    print('메인 상품 + 부대비용 계산 완료')
    main_df[[COL_FILE, COL_ITEM, COL_AMOUNT, '부대비용', '총비용']].head(20)
else:
    main_df = df[df['행_유형'] == '메인'].copy()
    print('금액 컬럼 없음 — 부대비용 계산 생략')
    main_df.head(20)

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 파일명별 메인 상품 수 집계
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

file_summary = (
    main_df.groupby(COL_FILE)
    .agg(
        메인상품수 = (COL_ITEM, 'count'),
        상품목록   = (COL_ITEM, lambda x: ' | '.join(x.dropna().astype(str).unique())),
    )
    .reset_index()
)

total_files  = len(file_summary)
single       = (file_summary['메인상품수'] == 1).sum()
multi        = (file_summary['메인상품수'] >= 2).sum()

print(f'전체 파일 수          : {total_files:,}개')
print(f'메인 상품 1개인 파일  : {single:,}개')
print(f'메인 상품 2개↑인 파일 : {multi:,}개  ← 목표')
print()
print('상품 수 분포:')
print(file_summary['메인상품수'].value_counts().sort_index().to_string())

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 다중 상품 파일 상세 목록 출력
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

multi_files = file_summary[file_summary['메인상품수'] >= 2].sort_values('메인상품수', ascending=False)

print(f'\n📦 메인 상품 2개 이상인 파일: 총 {len(multi_files)}개\n')
print('=' * 80)
for _, row in multi_files.iterrows():
    print(f"[{row[COL_FILE]}]  상품 {row['메인상품수']}개")
    for item in row['상품목록'].split(' | '):
        print(f'   • {item}')
    print()

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 결과 저장
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

from config import MULTI_ITEM_ANALYSIS_PATH

with pd.ExcelWriter(MULTI_ITEM_ANALYSIS_PATH, engine='openpyxl') as writer:
    # 시트1: 다중 상품 파일 목록
    multi_files.to_excel(writer, sheet_name='다중상품_파일목록', index=False)

    # 시트2: 메인 상품 + 부대비용 전체
    cols = [COL_FILE, COL_ITEM, '메인_idx']
    if '부대비용' in main_df.columns:
        cols += [COL_AMOUNT, '부대비용', '총비용']
    main_df[cols].to_excel(writer, sheet_name='메인상품_부대비용', index=False)

print(f'✅ 저장 완료: {MULTI_ITEM_ANALYSIS_PATH}')
print(f'   시트1 다중상품_파일목록 : {len(multi_files)}행')
print(f'   시트2 메인상품_부대비용 : {len(main_df)}행')